In [2]:
%pip install -q datasets huggingface_hub pillow matplotlib numpy pandas tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from __future__ import annotations

import math
import os
import random
import warnings
from collections import Counter, defaultdict
from itertools import islice
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from datasets import IterableDatasetDict, load_dataset
from datasets.features import ClassLabel
from huggingface_hub import HfApi

warnings.filterwarnings("default")
plt.style.use("seaborn-v0_8-whitegrid")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Figures directory: {FIGURES_DIR}")

c:\Users\Ipsita\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: i:\Inhouse project\UA-MTSeg-Rover-Navigation\UA-MTSeg-Rover-Navigation
Figures directory: i:\Inhouse project\UA-MTSeg-Rover-Navigation\UA-MTSeg-Rover-Navigation\figures


In [4]:
DATASET_ID = "hassanjbara/BASEPROD"
DATASET_CONFIG = None  # Set to a config name only if Hugging Face reports multiple configs.

MAX_SAMPLES_FOR_ANALYSIS = 60
MAX_SAMPLES_FOR_VISUAL_GRID = 6
MAX_PIXELS_PER_MODALITY_HIST = 250_000

HF_TOKEN = os.environ.get("HF_TOKEN") or None
print("HF_TOKEN detected in environment." if HF_TOKEN else "No HF_TOKEN detected; using public/anonymous access.")

HF_TOKEN detected in environment.


In [5]:
def safe_hf_dataset_info(dataset_id: str, token: Optional[str] = None):
    """Fetch lightweight Hugging Face Hub metadata without exposing credentials."""
    api = HfApi(token=token)
    try:
        return api.dataset_info(dataset_id)
    except Exception as exc:
        print(f"Could not read Hub metadata: {type(exc).__name__}: {exc}")
        return None

info = safe_hf_dataset_info(DATASET_ID, HF_TOKEN)
if info is not None:
    print(f"Dataset ID: {info.id}")
    print(f"private={getattr(info, 'private', None)}, gated={getattr(info, 'gated', None)}")
    print(f"Files reported by Hub: {len(getattr(info, 'siblings', []) or [])}")
else:
    print(f"Dataset ID: {DATASET_ID}")

Dataset ID: hassanjbara/BASEPROD
private=False, gated=False
Files reported by Hub: 110


In [6]:
def load_baseprod_streaming(dataset_id: str, config: Optional[str] = None, token: Optional[str] = None):
    kwargs = {"streaming": True}
    if token:
        kwargs["token"] = token
    if config:
        return load_dataset(dataset_id, config, **kwargs)
    return load_dataset(dataset_id, **kwargs)

try:
    streamed = load_baseprod_streaming(DATASET_ID, DATASET_CONFIG, HF_TOKEN)
except Exception as exc:
    raise RuntimeError(
        "Failed to load the dataset. If it is gated/private, set HF_TOKEN in the environment "
        "before launching VS Code/Jupyter. The token is intentionally never printed."
    ) from exc

print(type(streamed))
print(streamed)

<class 'datasets.dataset_dict.IterableDatasetDict'>
IterableDatasetDict({
    train: IterableDataset({
        features: ['color', 'depth', 'depth_16bit', 'thermal', 'thermal_rgb'],
        num_shards: 106
    })
})


In [7]:
def get_split_mapping(ds_obj):
    if isinstance(ds_obj, (IterableDatasetDict, dict)):
        return dict(ds_obj)
    return {"train": ds_obj}

splits = get_split_mapping(streamed)
print("Available splits:", list(splits.keys()))

split_rows = []
for split_name, split_ds in splits.items():
    features = getattr(split_ds, "features", None)
    split_rows.append({
        "split": split_name,
        "streaming_type": type(split_ds).__name__,
        "features_available": features is not None,
        "num_columns": len(features) if features else "unknown",
        "num_rows": "unknown in streaming mode unless provided by metadata",
    })
display(pd.DataFrame(split_rows))

Available splits: ['train']


,split,streaming_type,features_available,num_columns,num_rows
0,train,IterableDataset,True,5,unknown in streaming mode unless provided by m...


In [8]:
def feature_to_string(feature: Any) -> str:
    text = repr(feature)
    return text if len(text) <= 220 else text[:217] + "..."

for split_name, split_ds in splits.items():
    print(f"\n=== Features for split: {split_name} ===")
    features = getattr(split_ds, "features", None)
    if features is None:
        print("No declared features found; fields will be inferred from sampled records.")
        continue
    feature_df = pd.DataFrame([
        {"field": name, "feature_type": type(feature).__name__, "feature_repr": feature_to_string(feature)}
        for name, feature in features.items()
    ])
    display(feature_df)


=== Features for split: train ===


,field,feature_type,feature_repr
0,color,Image,"Image(mode=None, decode=True)"
1,depth,Image,"Image(mode=None, decode=True)"
2,depth_16bit,Image,"Image(mode=None, decode=True)"
3,thermal,Image,"Image(mode=None, decode=True)"
4,thermal_rgb,Image,"Image(mode=None, decode=True)"


In [ ]:
def take_stream_samples(split_ds, n: int) -> List[Dict[str, Any]]:
    samples = []
    for item in tqdm(islice(iter(split_ds), n), total=n, desc="Streaming sample records"):
        samples.append(item)
    return samples

PRIMARY_SPLIT = "train" if "train" in splits else next(iter(splits.keys()))
primary_ds = splits[PRIMARY_SPLIT]

samples = take_stream_samples(primary_ds, MAX_SAMPLES_FOR_ANALYSIS)
print(f"Loaded {len(samples)} streamed samples from split '{PRIMARY_SPLIT}'.")
if not samples:
    raise RuntimeError("No samples were loaded. Check dataset access and connectivity.")
print("Fields in first sampled record:")
print(list(samples[0].keys()))

Streaming sample records:   0%|          | 0/60 [00:00<?, ?it/s]'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/hassanjbara/BASEPROD/resolve/7a6a0d0a4e0ce95d3b5bbbaba3a066c4f18d67e3/data/train-00000-of-00106.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/hassanjbara/BASEPROD/resolve/7a6a0d0a4e0ce95d3b5bbbaba3a066c4f18d67e3/data/train-00000-of-00106.parquet
Retrying in 2s [Retry 2/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/hassanjbara/BASEPROD/resolve/7a6a0d0a4e0ce95d3b5bbbaba3a066c4f18d67e3/data/train-00000-of-00106.parquet
Retrying in 4s [Retry 3/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/hassanjbara/BASEPROD/resolve/7a6a0d0a4e0ce95d3b5bbbaba3a066c4f18d67e3/data/train-00000-of-00106.parquet
Retrying in 8s [Retry 4/5].
'The read operation timed out' thrown while 

In [ ]:
def value_to_array(value: Any) -> Optional[np.ndarray]:
    """Convert PIL, NumPy, or list image-like values to a NumPy array."""
    if value is None:
        return None
    if isinstance(value, Image.Image):
        return np.asarray(value)
    if isinstance(value, np.ndarray):
        return value
    if isinstance(value, list):
        try:
            arr = np.asarray(value)
            return arr if arr.ndim >= 2 else None
        except Exception:
            return None
    if isinstance(value, dict) and "array" in value:
        return value_to_array(value["array"])
    return None


def numeric_array(arr: Optional[np.ndarray]) -> Optional[np.ndarray]:
    if arr is None:
        return None
    try:
        arr = np.asarray(arr)
        if np.issubdtype(arr.dtype, np.number) or arr.dtype == bool:
            return arr.astype(np.float64, copy=False)
    except Exception:
        return None
    return None


def classify_field(field_name: str, arr: np.ndarray) -> str:
    lower = field_name.lower()
    ndim = arr.ndim
    channels = arr.shape[2] if ndim == 3 else 1

    if any(k in lower for k in ["label", "mask", "seg", "class", "annotation", "semantic"]):
        return "segmentation"
    if any(k in lower for k in ["thermal_rgb", "thermrgb", "thermal_color", "thermal_colour"]):
        return "thermal_rgb"
    if any(k in lower for k in ["thermal", "ir", "infrared", "temperature"]):
        return "thermal_rgb" if channels in (3, 4) else "thermal"
    if any(k in lower for k in ["depth", "range", "distance", "disp", "disparity"]):
        return "depth"
    if any(k in lower for k in ["rgb", "color", "colour", "image", "camera", "left", "right"]):
        return "rgb" if channels in (3, 4) else "grayscale_image"

    if ndim == 2:
        finite = arr[np.isfinite(arr)] if np.issubdtype(arr.dtype, np.number) else np.asarray([])
        if finite.size:
            unique_count = len(np.unique(finite[: min(finite.size, 50_000)]))
            if np.issubdtype(arr.dtype, np.integer) and unique_count <= 256:
                return "segmentation_candidate"
        return "single_channel_image"
    if ndim == 3 and channels in (3, 4):
        return "rgb_candidate"
    return "array"


def displayable_image(arr: np.ndarray, modality: str):
    """Prepare arrays for matplotlib display without modifying original data."""
    arr = np.asarray(arr)
    meta = {"cmap": None, "vmin": None, "vmax": None}

    if arr.ndim == 3 and arr.shape[2] == 4:
        arr = arr[:, :, :3]
    if arr.ndim == 3 and arr.shape[2] == 3:
        out = arr.astype(np.float64)
        out = out / 255.0 if np.nanmax(out) > 1.5 else out
        return np.clip(out, 0, 1), meta

    out = arr.squeeze().astype(np.float64)
    finite = out[np.isfinite(out)]
    if finite.size and not ("segmentation" in modality or "label" in modality):
        lo, hi = np.nanpercentile(finite, [1, 99])
        if math.isclose(lo, hi):
            lo, hi = float(np.nanmin(finite)), float(np.nanmax(finite))
        meta["vmin"], meta["vmax"] = lo, hi

    if "depth" in modality:
        meta["cmap"] = "viridis"
    elif "thermal" in modality:
        meta["cmap"] = "inferno"
    elif "segmentation" in modality or "label" in modality:
        meta["cmap"] = "tab20"
    else:
        meta["cmap"] = "gray"
    return out, meta

In [ ]:
field_records = []
for sample_idx, sample in enumerate(samples):
    for field_name, value in sample.items():
        arr = value_to_array(value)
        if arr is None:
            field_records.append({
                "sample_idx": sample_idx, "field": field_name,
                "python_type": type(value).__name__, "is_array_like": False,
                "shape": None, "dtype": None, "inferred_modality": "non_array",
            })
        else:
            field_records.append({
                "sample_idx": sample_idx, "field": field_name,
                "python_type": type(value).__name__, "is_array_like": True,
                "shape": tuple(arr.shape), "dtype": str(arr.dtype),
                "inferred_modality": classify_field(field_name, arr),
            })

field_df = pd.DataFrame(field_records)
display(field_df.head(30))
field_summary = (
    field_df.groupby(["field", "python_type", "is_array_like", "dtype", "inferred_modality"], dropna=False)
    .agg(samples_seen=("sample_idx", "count"), shapes=("shape", lambda x: sorted({str(v) for v in x if v is not None})[:8]))
    .reset_index()
    .sort_values(["is_array_like", "field"], ascending=[False, True])
)
display(field_summary)

In [ ]:
def detect_modalities(samples: List[Dict[str, Any]]) -> Dict[str, List[str]]:
    modality_to_fields = defaultdict(set)
    for sample in samples[: min(len(samples), 10)]:
        for field_name, value in sample.items():
            arr = value_to_array(value)
            if arr is None:
                continue
            modality = classify_field(field_name, arr)
            if modality == "rgb_candidate":
                modality = "rgb"
            if modality == "segmentation_candidate":
                modality = "segmentation"
            modality_to_fields[modality].add(field_name)
    return {k: sorted(v) for k, v in modality_to_fields.items()}

modalities = detect_modalities(samples)
print("Detected modality candidates:")
for modality, fields in modalities.items():
    print(f"- {modality}: {fields}")

preferred_order = ["rgb", "thermal_rgb", "depth", "thermal", "segmentation", "grayscale_image", "single_channel_image"]